# 모델 있고 없고 비교 — 실용성 검증

`train_emotion_classifier.ipynb`로 학습한 KoELECTRA 감정 분류기를 실제로 붙일 가치가 있는지 확인한다.

## 비교 대상 (4가지)

- **우리 모델** — `model_output/`에 저장된 우리 KoELECTRA(`predict.py`의 `EmotionPredictor`).
- **Gemini 제로샷** — `gemini-2.5-flash` 단독 제로샷 감정 분류.
  지금 `hyope-ai-server/llm_analyzer.py`의 프로덕션 프롬프트는 6종 감정 분류를 아예 하지 않고
  depressionScore 등 연속 점수만 낸다. 여기서는 "커스텀 모델 없이 LLM한테 감정 라벨까지 시키면
  어떨까"를 순수 제로샷 프롬프트로 재현한다.
- **GPT-5.6 Terra 제로샷** — OpenAI의 2026-07 신규 플래그십 패밀리 중 균형형 티어. 같은 제로샷
  프롬프트를 그대로 적용해 "최신 상용 LLM이면 우리 모델 없이도 충분하지 않을까"를 확인한다.
- **MelissaJ(공개모델)** — 우리와 똑같은 AI Hub 감성 대화 말뭉치 + 똑같은 6종 라벨로 학습된
  공개 KoELECTRA 모델. "같은 데이터로 남이 학습하면 대략 어느 정도 나오는지" 참고 기준점.

넷 다 같은 데이터로 채점해야 공정하므로 `train/evaluate.py`의 `split_aihub_holdout()`이 만드는
**보류 테스트셋**(학습에 전혀 쓰이지 않은 AI Hub 데이터, 1,015건)에서 뽑는다.

## 측정 항목

1. 정확도 / 매크로 F1 (레이블별)
2. 방식 간 예측 일치율(pairwise agreement), 서로 다르게 틀리는 사례
3. 건당 응답 시간 (실용성 — 배포 시 지연에 영향)

## 왜 전체 1,015건이 아니라 샘플인가

Gemini 무료 티어 호출은 시간이 많이 든다(아래 참고). 기본은 계층화 샘플 `SAMPLE_SIZE`건만 쓰고,
필요하면 아래 셀에서 값을 키워서 전체 보류셋으로 확장할 수 있다.

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI

# train/ 모듈 임포트 경로 (노트북은 notebooks/ 에서 실행된다고 가정)
TRAIN_DIR = Path.cwd().parent / "train"
sys.path.insert(0, str(TRAIN_DIR))

from config import ID2LABEL, LABEL2ID, LABELS, TrainConfig
from evaluate import split_aihub_holdout
from predict import EmotionPredictor

# GEMINI_API_KEY는 hyope-ai-server/.env에 있다 (별도 repo라 경로를 명시)
load_dotenv(Path.cwd().parent.parent / "hyope-ai-server" / ".env")

config = TrainConfig()
MODEL_PATH = config.output_dir  # 학습 노트북과 동일한 model_output/
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def get_gemini_client() -> OpenAI:
    return OpenAI(
        api_key=os.getenv("GEMINI_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )


def get_openai_client() -> OpenAI:
    return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


print(f"모델 경로: {MODEL_PATH}")
print(f"device: {DEVICE}")
print(f"GEMINI_API_KEY 설정됨: {bool(os.getenv('GEMINI_API_KEY'))}")
print(f"OPENAI_API_KEY 설정됨: {bool(os.getenv('OPENAI_API_KEY'))}")

## 1. 보류 테스트셋에서 계층화 샘플 뽑기

`split_aihub_holdout()`은 학습 노트북·`evaluate.py`와 **같은 seed**를 쓰므로,
여기서 뽑는 보류셋은 학습에 전혀 섞이지 않은 데이터임이 보장된다.

In [ ]:
from sklearn.model_selection import train_test_split

SAMPLE_SIZE = 30  # Gemini 무료 티어 분당 5회 제한 때문에 작게 잡음 (30건 ~= 6분). 늘리면 그만큼 오래 걸림

_, holdout = split_aihub_holdout(config)
print(f"보류 테스트셋 전체: {len(holdout)}건")

if SAMPLE_SIZE >= len(holdout):
    sample = holdout
else:
    sample, _ = train_test_split(
        holdout,
        train_size=SAMPLE_SIZE,
        random_state=config.seed,
        stratify=[s["label"] for s in holdout],
    )

texts = [s["text"] for s in sample]
true_labels = [s["label"] for s in sample]
print(f"평가 샘플: {len(sample)}건")
pd.Series([ID2LABEL[l] for l in true_labels]).value_counts().reindex(LABELS)

## 2. (A) 없는 경우 — Gemini 단독 제로샷 분류

프로덕션 프롬프트와 별개로, "감정 라벨만" 물어보는 최소 프롬프트를 사용한다.
`gemini-2.5-flash`는 내부적으로 thinking 토큰을 꽤 소모하므로 `max_tokens`를 넉넉히 줘야 한다
(100으로 테스트했다가 JSON이 중간에 잘려서 파싱 실패하는 걸 확인함 → 500으로 설정).

**주의: 무료 티어는 분당 5회 제한이다.** 실제로 동시 호출로 테스트했다가 12건 중 5건이
429(RESOURCE_EXHAUSTED)로 실패하는 걸 확인했다 — 이 실패를 그냥 오답 처리하면 Gemini 실력이
실제보다 나빠 보이는 착시가 생긴다. 그래서 동시 호출(ThreadPoolExecutor) 대신 분당 5회를
넘지 않게 자체 페이싱하는 순차 호출을 쓴다.

**실측 경고**: 검증 중 실제로는 "분당 5회"보다 훨씬 빡빡하게 걸렸다 — 1건 성공 후 서버가
`retryDelay`로 170~250초씩 대기를 요구하는 경우가 반복됐다 (같은 `GEMINI_API_KEY`를
`hyope-ai-server` 운영에서도 쓰고 있어서 그날 쿼터가 이미 소모됐을 수 있다). 코드는 서버가
알려주는 대기시간을 그대로 지키므로 결과는 정확하지만, 이 상태면 `SAMPLE_SIZE=30`도
1~2시간이 걸릴 수 있다. 처음엔 `SAMPLE_SIZE`를 10~15 정도로 낮춰 먼저 돌려보고,
너무 느리면 Gemini API 콘솔에서 그날 쿼터 사용량/티어를 확인해볼 것.

In [ ]:
import json
import re
from collections import deque

ZS_PROMPT = f"""다음 문장의 감정을 아래 6종 중 하나로만 분류하세요: {", ".join(LABELS)}

설명, markdown, 코드블록 없이 아래 JSON 형식만 반환하세요.
{{"emotion": "<6종 중 하나>"}}"""

RPM_LIMIT = 5  # 실측: 무료 티어 gemini-2.5-flash는 분당 5회 (429 에러로 확인)
_call_times: deque = deque()


def _wait_for_rate_limit():
    """직전 60초 안에 RPM_LIMIT회를 넘지 않도록 호출 전에 필요한 만큼 대기한다."""
    now = time.monotonic()
    while _call_times and now - _call_times[0] > 60:
        _call_times.popleft()
    if len(_call_times) >= RPM_LIMIT:
        time.sleep(max(60 - (now - _call_times[0]) + 1, 0))
    _call_times.append(time.monotonic())


def classify_with_gemini(client: OpenAI, text: str, max_retries: int = 4) -> dict:
    """단일 텍스트를 Gemini 제로샷으로 분류. 실패 시 emotion=None.

    분당 요청 수 제한에 걸리면(429) 서버가 알려주는 retryDelay만큼 기다렸다 재시도한다
    (그냥 오답 처리하면 rate limit과 실제 오분류를 구분할 수 없게 된다).
    """
    last_raw = ""
    latency = 0.0
    for attempt in range(max_retries + 1):
        _wait_for_rate_limit()
        start = time.perf_counter()
        try:
            resp = client.chat.completions.create(
                model="gemini-2.5-flash",
                messages=[
                    {"role": "system", "content": ZS_PROMPT},
                    {"role": "user", "content": text},
                ],
                temperature=0,
                max_tokens=500,
            )
        except Exception as e:
            latency = time.perf_counter() - start
            msg = str(e)
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                m = re.search(r"retryDelay['\"]?:\s*['\"]?(\d+)", msg)
                time.sleep(int(m.group(1)) + 1 if m else 60)
                continue
            if attempt < max_retries:
                continue
            return {"emotion": None, "latency": latency, "error": msg}
        latency = time.perf_counter() - start

        last_raw = resp.choices[0].message.content or ""
        cleaned = last_raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[-1]
            cleaned = cleaned.rsplit("```", 1)[0].strip()

        try:
            parsed = json.loads(cleaned)
            emotion = parsed.get("emotion")
            if emotion in LABEL2ID:
                return {"emotion": emotion, "latency": latency, "error": None}
        except json.JSONDecodeError:
            pass

        if attempt >= max_retries:
            return {"emotion": None, "latency": latency, "error": f"파싱 실패: {last_raw[:80]}"}
    return {"emotion": None, "latency": latency, "error": "unreachable"}

In [ ]:
from tqdm import tqdm

client = get_gemini_client()

gemini_results = [
    classify_with_gemini(client, t)
    for t in tqdm(texts, desc="Gemini 제로샷 (분당 5회 페이싱)")
]

gemini_failures = [r for r in gemini_results if r["emotion"] is None]
print(f"실패: {len(gemini_failures)}건 / {len(gemini_results)}건")
if gemini_failures:
    print("실패 예시:", gemini_failures[0]["error"])

## 3. (B) 있는 경우 — 우리 모델(KoELECTRA)

같은 `texts`로 `EmotionPredictor`를 돌린다. 로컬 배치 추론이라 API 호출보다 훨씬 빠를 것이다.

In [ ]:
predictor = EmotionPredictor(MODEL_PATH, device=DEVICE)

start = time.perf_counter()
ours_raw = predictor.predict_batch(texts)
ours_total_time = time.perf_counter() - start
ours_results = [{"emotion": r["emotion"], "latency": ours_total_time / len(texts)} for r in ours_raw]

print(f"우리 모델 총 소요: {ours_total_time:.2f}s ({len(texts)}건, 평균 {ours_total_time / len(texts) * 1000:.1f}ms/건)")

## 4. (A-2) 없는 경우 — OpenAI GPT-5.6 Terra 제로샷

같은 `ZS_PROMPT`를 OpenAI에도 그대로 적용한다 (2026-07 출시된 GPT-5.6 패밀리의 균형형 티어 —
플래그십 Sol보다 싸고, 단순 분류에는 굳이 Sol까지 안 써도 됨).

OpenAI는 유료 티어라 응답 헤더로 분당 500회까지 되는 걸 확인했다(`x-ratelimit-limit-requests: 500`)
— Gemini처럼 페이싱할 필요 없이 병렬로 돌린다. `reasoning_effort="none"`으로 불필요한 thinking
토큰 소모를 막는다(실측: `reasoning_tokens=0`). JSON 모드(`response_format`)를 지원해서
Gemini처럼 마크다운 코드블록 벗겨내는 처리도 필요 없다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

OPENAI_MODEL = "gpt-5.6-terra"


def classify_with_openai(client: OpenAI, text: str, max_retries: int = 3) -> dict:
    latency = 0.0
    for attempt in range(max_retries + 1):
        start = time.perf_counter()
        try:
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": ZS_PROMPT},
                    {"role": "user", "content": text},
                ],
                reasoning_effort="none",
                response_format={"type": "json_object"},
                max_completion_tokens=200,
            )
        except Exception as e:
            latency = time.perf_counter() - start
            if "429" in str(e) and attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            if attempt < max_retries:
                continue
            return {"emotion": None, "latency": latency, "error": str(e)}
        latency = time.perf_counter() - start

        raw = resp.choices[0].message.content or ""
        try:
            parsed = json.loads(raw)
            emotion = parsed.get("emotion")
            if emotion in LABEL2ID:
                return {"emotion": emotion, "latency": latency, "error": None}
        except json.JSONDecodeError:
            pass
        if attempt >= max_retries:
            return {"emotion": None, "latency": latency, "error": f"파싱 실패: {raw[:80]}"}
    return {"emotion": None, "latency": latency, "error": "unreachable"}


openai_client = get_openai_client()
openai_results = [None] * len(texts)
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(classify_with_openai, openai_client, t): i for i, t in enumerate(texts)}
    for future in tqdm(as_completed(futures), total=len(futures), desc="OpenAI GPT-5.6 Terra"):
        i = futures[future]
        openai_results[i] = future.result()

openai_failures = [r for r in openai_results if r["emotion"] is None]
print(f"실패: {len(openai_failures)}건 / {len(openai_results)}건")
if openai_failures:
    print("실패 예시:", openai_failures[0]["error"])

## 5. (B-2) 있는 경우 — 참고: 공개 모델(MelissaJ)

우리와 똑같은 AI Hub 감성 대화 말뭉치 + 똑같은 6종 라벨로 학습된 공개 KoELECTRA 모델
([MelissaJ/koelectra-emotion-6-emotion-base](https://huggingface.co/MelissaJ/koelectra-emotion-6-emotion-base)).
라벨 순서가 우리 `config.py`와 달라서(`id2label`이 다르게 배열됨) 모델 자체의 `id2label`로
텍스트를 얻은 뒤 우리 `LABEL2ID`로 다시 변환한다.

**주의**: 이 모델이 우리 보류 테스트셋의 일부를 자기 학습에 썼을 수도 있다(같은 원본 데이터셋을
각자 다르게 분할했을 가능성) — 그래서 여기 점수를 100% 공정한 비교로 보긴 어렵다. 그래도
"같은 데이터로 남이 학습하면 대략 어느 정도 나오는지" 참고 기준으로는 유용하다. 공개 README에
적힌 자체 검증 F1 macro는 약 0.60(3 epoch 기준, 그 이후는 과적합)이었다.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MELISSA_REPO = "MelissaJ/koelectra-emotion-6-emotion-base"

melissa_tokenizer = AutoTokenizer.from_pretrained(MELISSA_REPO)
melissa_model = AutoModelForSequenceClassification.from_pretrained(MELISSA_REPO)
melissa_model.to(DEVICE)
melissa_model.eval()
melissa_id2label = melissa_model.config.id2label

start = time.perf_counter()
melissa_results = []
with torch.no_grad():
    for i in range(0, len(texts), 32):
        batch = texts[i : i + 32]
        inputs = melissa_tokenizer(
            batch, return_tensors="pt", truncation=True, padding=True, max_length=256
        ).to(DEVICE)
        probs = torch.softmax(melissa_model(**inputs).logits, dim=-1)
        for p in probs:
            idx = int(p.argmax())
            melissa_results.append({"emotion": melissa_id2label[idx]})
melissa_total_time = time.perf_counter() - start
for r in melissa_results:
    r["latency"] = melissa_total_time / len(texts)

print(f"MelissaJ 모델 총 소요: {melissa_total_time:.2f}s (평균 {melissa_total_time / len(texts) * 1000:.1f}ms/건)")

## 6. 채점 — 정확도 / 매크로 F1

LLM 쪽이 실패(파싱 실패, rate limit 등)한 샘플은 오답 처리한다 — 실제 서비스에서 실패하면
그 값을 못 쓰는 것과 결과적으로 같기 때문.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

DUMMY = -1  # 실패(None)는 어떤 정답과도 안 맞는 더미 id로 처리 → 자동 오답


def to_pred_ids(results: list[dict]) -> list[int]:
    return [LABEL2ID.get(r["emotion"], DUMMY) for r in results]


PREDICTIONS = {
    "우리 모델": to_pred_ids(ours_results),
    "Gemini 제로샷": to_pred_ids(gemini_results),
    "GPT-5.6 Terra 제로샷": to_pred_ids(openai_results),
    "MelissaJ(공개모델)": to_pred_ids(melissa_results),
}
LATENCIES = {
    "우리 모델": [r["latency"] for r in ours_results],
    "Gemini 제로샷": [r["latency"] for r in gemini_results],
    "GPT-5.6 Terra 제로샷": [r["latency"] for r in openai_results],
    "MelissaJ(공개모델)": [r["latency"] for r in melissa_results],
}

label_ids = list(range(len(LABELS)))

rows = {}
for name, pred in PREDICTIONS.items():
    print(f"=== {name} ===")
    print(classification_report(true_labels, pred, labels=label_ids, target_names=LABELS, digits=3, zero_division=0))
    rows[name] = {
        "정확도": accuracy_score(true_labels, pred),
        "매크로 F1": f1_score(true_labels, pred, average="macro", zero_division=0),
        "평균 응답시간(ms)": float(np.mean(LATENCIES[name])) * 1000,
    }

summary = pd.DataFrame(rows).T
summary

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

_available = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _font in ["Malgun Gothic", "NanumGothic", "NanumBarunGothic", "AppleGothic"]:
    if _font in _available:
        plt.rcParams["font.family"] = _font
        break
else:
    print("경고: 한글 폰트를 찾지 못했습니다. 차트의 한글이 깨질 수 있습니다.")
plt.rcParams["axes.unicode_minus"] = False

# 4개 방식을 고정 순서 + 고정 색상으로 (우리 모델을 항상 맨 앞에 둬서 기준점 역할)
SERIES = ["우리 모델", "Gemini 제로샷", "GPT-5.6 Terra 제로샷", "MelissaJ(공개모델)"]
SERIES_COLORS = {
    "우리 모델": "#2a78d6",
    "Gemini 제로샷": "#1baf7a",
    "GPT-5.6 Terra 제로샷": "#eda100",
    "MelissaJ(공개모델)": "#a24f96",
}
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"


def style_axes(ax):
    ax.set_facecolor(SURFACE)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, labelcolor=INK_2)

In [ ]:
reports = {
    name: classification_report(true_labels, pred, labels=label_ids, target_names=LABELS, output_dict=True, zero_division=0)
    for name, pred in PREDICTIONS.items()
}
per_label_f1 = pd.DataFrame(
    {name: [reports[name][label]["f1-score"] for label in LABELS] for name in SERIES},
    index=LABELS,
)

fig, ax = plt.subplots(figsize=(9, 4.6))
y = np.arange(len(LABELS))
n = len(SERIES)
group_h = 0.7
bar_h = group_h / n
offsets = (np.arange(n) - (n - 1) / 2) * bar_h

for name, off in zip(SERIES, offsets):
    ax.barh(y - off, per_label_f1[name], height=bar_h * 0.9, color=SERIES_COLORS[name],
            label=name, edgecolor="white", linewidth=0.8)

ax.set_yticks(y, LABELS)
ax.set_xlim(0, 1.05)
style_axes(ax)
ax.set_title("레이블별 F1 — 4개 방식 비교", color=INK, loc="left", fontsize=12)
ax.legend(frameon=False, loc="lower right", labelcolor=INK_2, fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

metrics = ["정확도", "매크로 F1"]
x = np.arange(len(metrics))
n = len(SERIES)
width = 0.8 / n
offsets = (np.arange(n) - (n - 1) / 2) * width
for name, off in zip(SERIES, offsets):
    axes[0].bar(x + off, summary.loc[name, metrics], width=width * 0.9, color=SERIES_COLORS[name],
                label=name, edgecolor="white")
axes[0].set_xticks(x, metrics)
axes[0].set_ylim(0, 1.05)
style_axes(axes[0])
axes[0].set_title("정확도 / 매크로 F1", color=INK, loc="left", fontsize=11)
axes[0].legend(frameon=False, labelcolor=INK_2, fontsize=7.5, ncol=2)

latency_vals = [summary.loc[name, "평균 응답시간(ms)"] for name in SERIES]
axes[1].bar(range(len(SERIES)), latency_vals, color=[SERIES_COLORS[name] for name in SERIES], edgecolor="white")
axes[1].set_xticks(range(len(SERIES)), SERIES, rotation=20, ha="right", fontsize=8)
axes[1].set_yscale("log")
style_axes(axes[1])
axes[1].set_title("평균 응답시간 (ms/건, 로그축)", color=INK, loc="left", fontsize=11)
for i, v in enumerate(latency_vals):
    axes[1].text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=8, color=INK_2)

plt.tight_layout()
plt.show()

## 7. 방식들이 갈리는 지점

정답 기준 채점 말고, **서로 다르게 예측한 사례**를 보면 실제로 뭐가 다른지 감이 온다.
특히 "우리 모델만 맞고 나머지(Gemini/GPT-5.6/공개모델)는 다 틀린 경우"와 그 반대 경우가
우리 모델을 더 개발할 가치가 있는지에 대한 가장 직접적인 증거다.

In [ ]:
# 방식 간 pairwise 예측 일치율
agree_matrix = pd.DataFrame(index=SERIES, columns=SERIES, dtype=float)
for a in SERIES:
    for b in SERIES:
        agree_matrix.loc[a, b] = np.mean([x == y for x, y in zip(PREDICTIONS[a], PREDICTIONS[b])])
print("방식 간 예측 일치율:")
display(agree_matrix.round(2))

diff_df = pd.DataFrame({"텍스트": texts, "정답": [ID2LABEL[t] for t in true_labels]})
for name in SERIES:
    diff_df[name] = [ID2LABEL.get(p, "(실패)") for p in PREDICTIONS[name]]

baselines = [n for n in SERIES if n != "우리 모델"]
ours_correct = diff_df["우리 모델"] == diff_df["정답"]
all_baselines_wrong = diff_df[baselines].ne(diff_df["정답"], axis=0).all(axis=1)
only_ours_correct = diff_df[ours_correct & all_baselines_wrong]

all_baselines_correct = diff_df[baselines].eq(diff_df["정답"], axis=0).all(axis=1)
ours_wrong = diff_df["우리 모델"] != diff_df["정답"]
ours_uniquely_wrong = diff_df[ours_wrong & all_baselines_correct]

print(f"우리 모델만 맞춘 사례: {len(only_ours_correct)}건 / 나머지 전부 맞췄는데 우리만 틀린 사례: {len(ours_uniquely_wrong)}건")
with pd.option_context("display.max_colwidth", 45):
    display(only_ours_correct)
    display(ours_uniquely_wrong)

## 8. 실용성 판단 가이드

숫자를 채운 뒤 아래 기준으로 판단한다 (임의 기준이니 팀 기준에 맞게 조정):

- **Gemini·GPT-5.6과 매크로 F1 차이가 크지 않다면(예: 0.03 미만)** → 어차피 Gemini는 통화마다
  호출하므로, 같은 호출에 감정 분류 필드를 하나 추가하는 게 더 간단할 수 있다.
  별도 모델 배포/유지보수 비용을 아낀다.
- **MelissaJ(같은 데이터로 학습한 공개 모델, 자체 보고 F1 macro ≈ 0.60)보다 우리 모델이 뚜렷이
  높다면** → 우리 학습 파이프라인(3개 데이터 소스 결합 등)이 실제로 효과가 있었다는 뜻.
  반대로 비슷하거나 낮다면 굳이 우리가 재학습할 필요 없이 공개 모델을 가져다 쓰는 것도 고려할 만하다.
- **우리 모델이 LLM들보다 뚜렷하게 더 정확하다면** → 노인 발화 도메인(AI Hub 데이터)에 특화
  학습한 효과가 실제로 있다는 뜻. 응답시간 차이(로컬 추론이 훨씬 빠름)까지 고려하면 배포 가치가 커진다.
- **LLM 쪽 실패율(파싱 실패, rate limit 등)이 유의미하다면** → 프로덕션에서도 같은 실패가 날 수
  있다는 뜻이니 이것만으로도 로컬 모델을 쓸 이유가 된다 (실패 시 폴백이 없으면 그 필드가 통째로 빠진다).
- **`SAMPLE_SIZE`를 키워서(예: 전체 1,015건) 다시 돌리면** 신뢰구간이 좁아져 판단이 더 명확해진다.

이 노트북은 판단에 필요한 숫자를 만드는 것까지가 목적이고, 최종 결정(더 개발할지/배포할지)은
팀 회의에서.

## 9. 독립 테스트셋 검증 — GoEmotions-Korean

MelissaJ가 AI Hub 감성 대화로 학습됐기 때문에, 우리 AI Hub 보류셋으로 채점하면 MelissaJ가
답을 이미 알고 있을 위험이 크다 (자체 보고 F1 macro ≈ 0.60인데 우리 테스트에서 0.966이 나온 게
그 증거). 그래서 **AI Hub도, 우리 학습에 쓴 KOTE/KorEmpatheticDialogues도 아닌** 완전히 독립된
테스트셋으로 다시 채점한다.

**[GoEmotions-Korean](https://github.com/monologg/GoEmotions-Korean)** — Reddit 댓글 58,000건을
28종 감정으로 라벨링한 원본(GoEmotions)을 구글 번역으로 한국어화한 데이터. AI Hub와 무관하고
우리 학습에도 전혀 안 쓰였다.

(참고: `AI_model_research.md`에 있던 다른 후보 KEmoFact는 제외했다 — 알고보니 EmpatheticDialogues
기반이라, 우리가 학습에 쓴 KorEmpatheticDialogues와 같은 계열이라서 이번엔 반대로 우리 쪽에
유리하게 샐 위험이 있다.)

**주의**: Reddit 댓글 + 기계번역이라 어색한 문장이 많고 노인 발화체와는 register가 다르다
(원본 README가 "번역 품질이 좋지 않다"고 스스로 경고함). 그래도 "AI Hub를 안 본 상태에서의
진짜 실력"을 보는 용도로는 유효하다. 이번 라운드는 Gemini 무료 티어가 너무 오래 걸려서 빼고
우리 모델 / GPT-5.6 Terra / MelissaJ 셋만 비교한다.

In [ ]:
import csv
import io
import urllib.request

GOEMOTIONS_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring", "confusion",
    "curiosity", "desire", "disappointment", "disapproval", "disgust", "embarrassment",
    "excitement", "fear", "gratitude", "grief", "joy", "love", "nervousness", "optimism",
    "pride", "realization", "relief", "remorse", "sadness", "surprise", "neutral",
]

# 우리 6종과 최대한 개념이 겹치는 것만 매핑, 애매한 건(desire/curiosity/caring/neutral 등) 스킵
GOEMOTIONS_TO_6 = {
    "anger": "분노", "annoyance": "분노", "disgust": "분노", "disapproval": "분노",
    "joy": "기쁨", "amusement": "기쁨", "admiration": "기쁨", "excitement": "기쁨",
    "gratitude": "기쁨", "love": "기쁨", "optimism": "기쁨", "pride": "기쁨",
    "approval": "기쁨", "relief": "기쁨",
    "fear": "불안", "nervousness": "불안",
    "embarrassment": "당황", "surprise": "당황", "confusion": "당황",
    "sadness": "슬픔", "grief": "슬픔", "remorse": "슬픔",
    "disappointment": "상처",
}

GOEMOTIONS_URL = "https://raw.githubusercontent.com/monologg/GoEmotions-Korean/master/data/test.tsv"

with urllib.request.urlopen(GOEMOTIONS_URL) as resp:
    content = resp.read().decode("utf-8")

goemo_samples = []
for text, label_str in csv.reader(io.StringIO(content), delimiter="	"):
    text = text.strip()
    if not text:
        continue
    for idx in label_str.split(","):
        mapped = GOEMOTIONS_TO_6.get(GOEMOTIONS_LABELS[int(idx)])
        if mapped:
            goemo_samples.append({"text": text, "label": LABEL2ID[mapped]})
            break

print(f"GoEmotions-Korean 매핑 성공: {len(goemo_samples)}건")
pd.Series([ID2LABEL[s["label"]] for s in goemo_samples]).value_counts().reindex(LABELS)

In [ ]:
import random

random.seed(config.seed)

SAMPLE_SIZE_2 = 30
per_class_target = max(1, SAMPLE_SIZE_2 // len(LABELS))

by_label = {i: [s for s in goemo_samples if s["label"] == i] for i in range(len(LABELS))}
goemo_sample = []
for i, group in by_label.items():
    random.shuffle(group)
    goemo_sample.extend(group[:per_class_target])
random.shuffle(goemo_sample)

texts2 = [s["text"] for s in goemo_sample]
true_labels2 = [s["label"] for s in goemo_sample]
print("클래스별 확보 샘플 수:", {ID2LABEL[i]: min(len(by_label[i]), per_class_target) for i in range(len(LABELS))})
print(f"평가 샘플: {len(goemo_sample)}건")

### 세 방식 실행

앞서 정의한 `predictor`(우리 모델), `classify_with_openai`(GPT-5.6 Terra),
`melissa_tokenizer`/`melissa_model`(공개모델)을 그대로 재사용한다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# 우리 모델
start = time.perf_counter()
ours_raw2 = predictor.predict_batch(texts2)
ours_total_time2 = time.perf_counter() - start
ours_results2 = [{"emotion": r["emotion"], "latency": ours_total_time2 / len(texts2)} for r in ours_raw2]
print(f"우리 모델: {ours_total_time2:.2f}s (평균 {ours_total_time2 / len(texts2) * 1000:.1f}ms/건)")

# GPT-5.6 Terra 제로샷
openai_client2 = get_openai_client()
openai_results2 = [None] * len(texts2)
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(classify_with_openai, openai_client2, t): i for i, t in enumerate(texts2)}
    for future in tqdm(as_completed(futures), total=len(futures), desc="GPT-5.6 Terra (GoEmotions)"):
        i = futures[future]
        openai_results2[i] = future.result()
openai_failures2 = [r for r in openai_results2 if r["emotion"] is None]
print(f"GPT-5.6 Terra 실패: {len(openai_failures2)}건 / {len(openai_results2)}건")

# MelissaJ(공개모델)
start = time.perf_counter()
melissa_results2 = []
with torch.no_grad():
    for i in range(0, len(texts2), 32):
        batch = texts2[i : i + 32]
        inputs = melissa_tokenizer(
            batch, return_tensors="pt", truncation=True, padding=True, max_length=256
        ).to(DEVICE)
        probs = torch.softmax(melissa_model(**inputs).logits, dim=-1)
        for p in probs:
            idx = int(p.argmax())
            melissa_results2.append({"emotion": melissa_id2label[idx]})
melissa_total_time2 = time.perf_counter() - start
for r in melissa_results2:
    r["latency"] = melissa_total_time2 / len(texts2)
print(f"MelissaJ: {melissa_total_time2:.2f}s (평균 {melissa_total_time2 / len(texts2) * 1000:.1f}ms/건)")

In [ ]:
SERIES2 = ["우리 모델", "GPT-5.6 Terra 제로샷", "MelissaJ(공개모델)"]

PREDICTIONS2 = {
    "우리 모델": to_pred_ids(ours_results2),
    "GPT-5.6 Terra 제로샷": to_pred_ids(openai_results2),
    "MelissaJ(공개모델)": to_pred_ids(melissa_results2),
}

rows2 = {}
for name, pred in PREDICTIONS2.items():
    print(f"=== {name} (GoEmotions-Korean) ===")
    print(classification_report(true_labels2, pred, labels=label_ids, target_names=LABELS, digits=3, zero_division=0))
    rows2[name] = {
        "정확도": accuracy_score(true_labels2, pred),
        "매크로 F1": f1_score(true_labels2, pred, average="macro", zero_division=0),
    }

summary2 = pd.DataFrame(rows2).T
summary2

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
metrics = ["정확도", "매크로 F1"]
x = np.arange(len(metrics))
n = len(SERIES2)
width = 0.8 / n
offsets = (np.arange(n) - (n - 1) / 2) * width
for name, off in zip(SERIES2, offsets):
    ax.bar(x + off, summary2.loc[name, metrics], width=width * 0.9, color=SERIES_COLORS[name],
           label=name, edgecolor="white")
ax.set_xticks(x, metrics)
ax.set_ylim(0, 1.05)
style_axes(ax)
ax.set_title("GoEmotions-Korean(독립 테스트셋) 비교", color=INK, loc="left", fontsize=12)
ax.legend(frameon=False, labelcolor=INK_2, fontsize=8)
plt.tight_layout()
plt.show()